In [ ]:
import os
import zipfile
import pickle
import pandas as pd
from tqdm import tqdm  # Import tqdm for the progress bar

def load_dataframe_from_zip(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        # Iterate through all files in the zip
        for file_name in zip_ref.namelist():
            if file_name.endswith('.pkl'):
                with zip_ref.open(file_name) as file:
                    # Load the data from the pickle file
                    data = pickle.load(file)
                    
                    # Check if the data is a list, and if so, convert it to a DataFrame
                    if isinstance(data, list):
                        df = pd.DataFrame(data, columns=['cik', 'date', 'note_text'])
                        df['item_7'] = df['note_text'].apply(extract_item_7)
                        df['risk_factors'] = df['note_text'].apply(extract_risk_factors)

                        return df[['cik', 'date','item_7','risk_factors']]
                    else:
                        print(f"Warning: The file {file_name} does not contain a list or DataFrame.")
    return None

def combine_pickles_in_folders(folders):
    combined_df = pd.DataFrame()
    
    # Loop through each folder
    for folder in folders:
        for root, dirs, files in os.walk(folder):
            # Add tqdm to show progress when iterating over the files
            for file in tqdm(files, desc=f"Processing files in {folder}", unit="file"):
                if file.endswith('.zip'):
                    zip_path = os.path.join(root, file)
                    # print(f"Processing {zip_path}")
                    
                    # Load the dataframe from the zip file
                    df = load_dataframe_from_zip(zip_path)
                    if df is not None:
                        combined_df = pd.concat([combined_df, df], ignore_index=True)
    
    return combined_df

# Define the folders to search
folders = ['processed_data', 'processed_data_new']

# Combine the dataframes
combined_dataframe = combine_pickles_in_folders(folders)

# Save the combined dataframe as a Parquet file
if not combined_dataframe.empty:
    combined_dataframe.to_parquet('combined_dataframe.parquet', compression='gzip')
    print("DataFrames combined and saved as 'combined_dataframe.parquet'")
else:
    print("No data found to combine.")


Processing files in processed_data:   3%|█▏                                          | 7/266 [00:14<07:52,  1.83s/file]

In [3]:
def extract_risk_factors(text):
    # Normalize the text to handle different cases and line breaks
    normalized_text = text.lower()

    # Define the main regex pattern to extract the content between "Item 1A. Risk Factors" and the subsequent section
    pattern = r'[\r\n]+\s*item[\s\n]*1[\s\n]*a[\s\n]*\.?[\s\n]*-?[\s\n]*risks?\s*factors?[\s\n]*(.*?)(?=[\r\n]+\s*item[\s\n]*1[\s\n]*b[\s\n]*\.?[\s\n]*?-?|[\r\n]+\s*item[\s\n]*2[\s\n]*-?|\Z)'

    # Search for the pattern in the normalized text
    matches = list(re.finditer(pattern, normalized_text, re.DOTALL | re.IGNORECASE))
    
    # Extract the desired section if found
    sections = []
    for match in matches:
        content_start, content_end = match.start(1), match.end(1)
        sections.append(text[content_start:content_end].strip())
    
    # If the main pattern isn't found, check if "Item 1A. Risk Factors" appears standalone
    if not sections:
        standalone_pattern = r'[\r\n]+\s*item[\s\n]*1[\s\n]*a[\s\n]*\.?[\s\n]*-?[\s\n]*risks?\s*factors?[\s\n]*'

        standalone_match = re.search(standalone_pattern, normalized_text)
        if standalone_match:
            sections.append(text[standalone_match.start(1):].strip())

    return sections if sections else None

In [17]:
import re

def extract_item_7(text):
    # Normalize for consistent matching
    normalized_text = text.lower()

    # Pattern: match "Item 7" until the next line that starts with "Item" followed by a number and optional letter
    pattern = r'(item\s+7[^a-z0-9]*[\s\S]*?)(?=\n\s*item\s+\d+[a-z]?\b)'

    match = re.search(pattern, normalized_text, re.IGNORECASE)
    if match:
        return match.group(1).strip()
    return None
